**《Elucidating the Design Space of Diffusion-Based Generative Models》**：

https://arxiv.org/pdf/2206.00364


## 一、论文第 1–2 节：为什么能够统一不同扩散模型？

### 1.1 先定义论文使用的概率分布

为了避免和 DDPM 的时间下标混淆，我们统一记：

- $y\in\mathbb R^d$：真实数据。
- $p_{\mathrm{data}}(y)$：真实数据分布。
- $x\in\mathbb R^d$：加噪数据。
- $\sigma>0$：噪声标准差。
- $\epsilon\sim\mathcal N(0,I)$：标准高斯噪声。
- $p(x;\sigma)$：噪声水平为 $\sigma$ 时的边缘分布。

定义：

$$
x=y+\sigma\epsilon
$$

那么：

$$
\boxed{
p(x;\sigma)=
\int p_{\mathrm{data}}(y)
\mathcal N(x;y,\sigma^2I)\,dy
}
$$

注意：这里没有定义前向过程中的相邻转移概率。

我们只定义了**每一个噪声水平下的边缘分布**。

这与之前讨论 DDIM 时的思路非常接近：我们可以先指定一族边缘分布，再研究如何在这些分布之间移动。

EDM 的目标是找到一个连续动力系统，使得：

$$
x(t_a)\sim p(x;\sigma(t_a))
$$

经过演化后：

$$
x(t_b)\sim p(x;\sigma(t_b))
$$

这里 $\sigma(t)$ 是噪声时间表。

---

### 1.2 论文公式 (1)：Probability Flow ODE

原文公式 (1)：

$$
\boxed{
\frac{dx}{dt}
=
-\dot\sigma(t)\sigma(t)
\nabla_x\log p(x;\sigma(t))
}
\tag{1}
$$

其中：

$$
\dot\sigma(t)=\frac{d\sigma(t)}{dt}
$$

我们不能直接把这个公式当作定义，而应该推导为什么它能够保持正确的边缘分布。

这里使用论文附录 B.1、B.2 的思想。

#### 第一步：从高斯卷积推导热方程

定义：

$$
G_\sigma(x-y)
=
\frac1{(2\pi\sigma^2)^{d/2}}
\exp\left(
-\frac{\|x-y\|^2}{2\sigma^2}
\right)
$$

那么：

$$
p(x;\sigma)
=
\int p_{\mathrm{data}}(y)G_\sigma(x-y)\,dy
$$

先对高斯核关于 $\sigma$ 求导。

取对数：

$$
\log G_\sigma
=
-\frac d2\log(2\pi)
-d\log\sigma
-\frac{\|x-y\|^2}{2\sigma^2}
$$

因此：

$$
\frac{\partial\log G_\sigma}{\partial\sigma}
=
-\frac d\sigma
+
\frac{\|x-y\|^2}{\sigma^3}
$$

得到：

$$
\frac{\partial G_\sigma}{\partial\sigma}
=
G_\sigma
\left(
-\frac d\sigma+
\frac{\|x-y\|^2}{\sigma^3}
\right)
$$

再计算关于 $x$ 的 Laplacian：

$$
\Delta_x G_\sigma
=
G_\sigma
\left(
-\frac d{\sigma^2}
+
\frac{\|x-y\|^2}{\sigma^4}
\right)
$$

其中：

$$
\Delta_x=\sum_{j=1}^d\frac{\partial^2}{\partial x_j^2}
$$

所以：

$$
\boxed{
\frac{\partial G_\sigma}{\partial\sigma}
=
\sigma\Delta_xG_\sigma
}
$$

对数据分布积分：

$$
\frac{\partial p(x;\sigma)}{\partial\sigma}
=
\int p_{\mathrm{data}}(y)
\frac{\partial G_\sigma(x-y)}{\partial\sigma}\,dy
$$

代入：

$$
=
\sigma
\int p_{\mathrm{data}}(y)
\Delta_xG_\sigma(x-y)\,dy
$$

在适当的正则条件下交换微分与积分：

$$
\boxed{
\frac{\partial p(x;\sigma)}{\partial\sigma}
=
\sigma\Delta_xp(x;\sigma)
}
$$

现在考虑：

$$
\sigma=\sigma(t)
$$

根据链式法则：

$$
\frac{\partial p_t(x)}{\partial t}
=
\dot\sigma(t)
\frac{\partial p(x;\sigma)}{\partial\sigma}
$$

得到：

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
=
\sigma(t)\dot\sigma(t)\Delta_xp_t(x)
}
\tag{A}
$$

这就是加性高斯噪声分布随时间演化满足的热方程。

#### 第二步：寻找具有相同边缘分布的 ODE

对于确定性动力系统：

$$
\frac{dx}{dt}=v(x,t)
$$

其概率密度满足连续性方程：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
-\nabla_x\cdot(p_tv)
}
\tag{B}
$$

其中 $v(x,t)$ 是速度场。

我们希望式 (B) 等于刚才得到的式 (A)：

$$
-\nabla_x\cdot(p_tv)
=
\sigma\dot\sigma\Delta_xp_t
$$

注意恒等式：

$$
p_t\nabla_x\log p_t=\nabla_xp_t
$$

因此，如果选择：

$$
v(x,t)
=
-\sigma\dot\sigma\nabla_x\log p_t(x)
$$

就有：

$$
-\nabla_x\cdot(p_tv)
=
\sigma\dot\sigma
\nabla_x\cdot(p_t\nabla_x\log p_t)
$$

$$
=
\sigma\dot\sigma\Delta_xp_t
$$

恰好满足要求。

于是得到：

$$
\boxed{
\frac{dx}{dt}
=
-\sigma(t)\dot\sigma(t)
\nabla_x\log p(x;\sigma(t))
}
$$

这就是论文公式 (1)。

**这个推导最重要的结论是：只要使用真实 Score，这个 ODE 就能够在指定的高斯平滑边缘分布之间运输概率质量。**

它不是凭空设计的速度场，而是由热方程和概率连续性方程共同确定的一种选择。

同时要注意：满足连续性方程的速度场不一定唯一。EDM 使用的是由 Probability Flow 构造得到的这一种。

---

## 二、论文公式 (2)、(3)：为什么预测干净数据就能得到 Score？

这部分是 EDM 网络参数化的理论基础。

论文先定义去噪器：

$$
D(x;\sigma)
$$

它的训练目标为：

$$
\boxed{
\mathcal L_{\mathrm{denoise}}
=
\mathbb E_{y\sim p_{\mathrm{data}}}
\mathbb E_{n\sim\mathcal N(0,\sigma^2I)}
\left[
\|D(y+n;\sigma)-y\|^2
\right]
}
\tag{2}
$$

其中 $n=\sigma\epsilon$。

我们先求这个目标的理论最优解。

### 2.1 MSE 最优解是条件期望

给定加噪样本：

$$
x=y+n
$$

我们希望最小化：

$$
\mathbb E[\|D(x)-y\|^2\mid x]
$$

令：

$$
m(x)=\mathbb E[y\mid x]
$$

把误差拆成：

$$
D(x)-y
=
[D(x)-m(x)]+[m(x)-y]
$$

平方并取条件期望：

$$
\begin{aligned}
\mathbb E[\|D(x)-y\|^2\mid x]
={}&
\|D(x)-m(x)\|^2\\
&+\mathbb E[\|m(x)-y\|^2\mid x]\\
&+2(D(x)-m(x))^\top
\mathbb E[m(x)-y\mid x]
\end{aligned}
$$

由于：

$$
\mathbb E[m(x)-y\mid x]=0
$$

交叉项消失。

第二项与 $D$ 无关，因此最优解为：

$$
\boxed{
D^*(x;\sigma)=\mathbb E[y\mid x]
}
$$

### 2.2 从条件期望推导 Score

现在计算：

$$
\nabla_x\log p(x;\sigma)
=
\frac{\nabla_xp(x;\sigma)}{p(x;\sigma)}
$$

根据：

$$
p(x;\sigma)
=
\int p_{\mathrm{data}}(y)
G_\sigma(x-y)\,dy
$$

求梯度：

$$
\nabla_xp(x;\sigma)
=
\int p_{\mathrm{data}}(y)
\nabla_xG_\sigma(x-y)\,dy
$$

高斯核满足：

$$
\nabla_xG_\sigma(x-y)
=
-\frac{x-y}{\sigma^2}G_\sigma(x-y)
$$

因此：

$$
\nabla_xp(x;\sigma)
=
\int
p_{\mathrm{data}}(y)
\frac{y-x}{\sigma^2}
G_\sigma(x-y)\,dy
$$

两边除以 $p(x;\sigma)$：

$$
\nabla_x\log p(x;\sigma)
=
\frac1{\sigma^2}
\int(y-x)
\frac{p_{\mathrm{data}}(y)G_\sigma(x-y)}
{p(x;\sigma)}
\,dy
$$

根据贝叶斯公式：

$$
p(y\mid x;\sigma)
=
\frac{p_{\mathrm{data}}(y)G_\sigma(x-y)}
{p(x;\sigma)}
$$

所以：

$$
\nabla_x\log p(x;\sigma)
=
\frac1{\sigma^2}
\int(y-x)p(y\mid x;\sigma)\,dy
$$

整理：

$$
=
\frac{\mathbb E[y\mid x]-x}{\sigma^2}
$$

由于：

$$
D^*(x;\sigma)=\mathbb E[y\mid x]
$$

最终：

$$
\boxed{
\nabla_x\log p(x;\sigma)
=
\frac{D^*(x;\sigma)-x}{\sigma^2}
}
\tag{3}
$$

这就是论文公式 (3)。

### 2.3 为什么这个结论重要？

因为我们原本需要学习：

$$
\nabla_x\log p(x;\sigma)
$$

但是现在只需要训练一个去噪网络：

$$
D_\theta(x;\sigma)
$$

然后通过：

$$
\boxed{
s_\theta(x,\sigma)
=
\frac{D_\theta(x;\sigma)-x}{\sigma^2}
}
$$

就能构造 Score 的估计。

再代入公式 (1)：

$$
\frac{dx}{dt}
=
-\sigma\dot\sigma
\frac{D_\theta-x}{\sigma^2}
$$

得到：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot\sigma}{\sigma}
(x-D_\theta)
}
$$

这一步把理论上的 Score ODE 转换成了可以实际运行的去噪网络采样器。

**注意，公式 (3) 对理论最优去噪器 $D^*$ 是精确恒等式；对有限容量、有限训练数据的 $D_\theta$ 则是近似关系。**

---

## 三、论文公式 (4)：为什么还要引入 $s(t)$？

前面我们得到的 ODE 只包含：

$$
\sigma(t)
$$

但 EDM 进一步引入了：

$$
s(t)
$$

它表示随时间变化的信号缩放系数。

这是统一 VP、VE、DDIM 的关键。

### 3.1 定义缩放后的状态

令：

$$
\hat x(t)
$$

表示未缩放状态，满足：

$$
\frac{d\hat x}{dt}
=
-\sigma\dot\sigma
\nabla_{\hat x}\log p(\hat x;\sigma)
$$

现在定义：

$$
\boxed{
x(t)=s(t)\hat x(t)
}
$$

其中 $s(t)>0$。

对时间求导：

$$
\frac{dx}{dt}
=
\dot s(t)\hat x+
s(t)\frac{d\hat x}{dt}
$$

代入：

$$
\hat x=\frac{x}{s}
$$

得到：

$$
\frac{dx}{dt}
=
\frac{\dot s}{s}x+
s\frac{d\hat x}{dt}
$$

再代入原来的 ODE：

$$
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s\sigma\dot\sigma
\nabla_{\hat x}\log p(\hat x;\sigma)
$$

最后把：

$$
\hat x=\frac{x}{s}
$$

代回：

$$
\boxed{
\frac{dx}{dt}
=
\frac{\dot s(t)}{s(t)}x
-
s(t)\sigma(t)\dot\sigma(t)
\nabla_{\hat x}
\log p\left(
\frac{x}{s(t)};\sigma(t)
\right)
}
\tag{4}
$$

这里梯度是对未缩放变量 $\hat x$ 求的，随后在 $\hat x=x/s(t)$ 处取值。

这就是 EDM 的广义 ODE。

### 3.2 将去噪器代入广义 ODE

根据公式 (3)：

$$
\nabla_{\hat x}\log p(\hat x;\sigma)
=
\frac{D_\theta(\hat x;\sigma)-\hat x}{\sigma^2}
$$

所以：

$$
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
s\sigma\dot\sigma
\frac{
D_\theta(x/s;\sigma)-x/s
}{
\sigma^2
}
$$

整理：

$$
\frac{dx}{dt}
=
\frac{\dot s}{s}x
-
\frac{s\dot\sigma}{\sigma}D_\theta(x/s;\sigma)
+
\frac{\dot\sigma}{\sigma}x
$$

得到：

$$
\boxed{
\frac{dx}{dt}
=
\left(
\frac{\dot s}{s}
+
\frac{\dot\sigma}{\sigma}
\right)x
-
\frac{s\dot\sigma}{\sigma}
D_\theta\left(\frac{x}{s};\sigma\right)
}
$$

这正是论文 Algorithm 1 中实际计算速度的表达式。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

### 3.3 VP、VE 为什么都能放进这个公式？

以 VP-SDE 为例。

原来的 VP 加噪公式：

$$
x_t=\sqrt{\bar\alpha_t}y+
\sqrt{1-\bar\alpha_t}\epsilon
$$

我们可以写成：

$$
x_t=
s(t)[y+\sigma(t)\epsilon]
$$

只要选择：

$$
\boxed{
s(t)=\sqrt{\bar\alpha_t}
}
$$

$$
\boxed{
\sigma(t)=
\sqrt{\frac{1-\bar\alpha_t}{\bar\alpha_t}}
}
$$

就有：

$$
s(t)[y+\sigma(t)\epsilon]
=
\sqrt{\bar\alpha_t}y+
\sqrt{1-\bar\alpha_t}\epsilon
$$

与 VP 完全一致。

而 VE 可以选择：

$$
\boxed{s(t)=1}
$$

只保留噪声水平随时间变化。

这说明：

**VP 和 VE 的区别，在 EDM 的统一表示下，很大程度上体现为 $\sigma(t)$ 和 $s(t)$ 的不同选择。**

但这不意味着它们原始的随机过程或训练细节完全相同。

---

## 四、论文第 3 节：为什么 EDM 选择 $\sigma(t)=t,\ s(t)=1$？

这是论文采样器设计的核心。

如果选择：

$$
s(t)=1
$$

那么：

$$
\dot s(t)=0
$$

广义 ODE 简化为：

$$
\frac{dx}{dt}
=
\frac{\dot\sigma(t)}{\sigma(t)}
[x-D_\theta(x;\sigma(t))]
$$

进一步选择：

$$
\sigma(t)=t
$$

于是：

$$
\dot\sigma(t)=1
$$

得到：

$$
\boxed{
\frac{dx}{dt}
=
\frac{x-D_\theta(x;t)}{t}
}
$$

为什么论文认为这种选择有优势？

### 4.1 从 Euler 的几何意义理解

Euler 更新：

$$
x_{\mathrm{next}}
=
x_t+h\frac{x_t-D_\theta(x_t;t)}{t}
$$

如果直接取：

$$
h=-t
$$

也就是从当前时间一步积分到零：

$$
x_{\mathrm{next}}
=
x_t-t\frac{x_t-D_\theta(x_t;t)}{t}
$$

所以：

$$
\boxed{
x_{\mathrm{next}}=D_\theta(x_t;t)
}
$$

这说明：

**EDM 的 ODE 在当前点的切线方向，正好指向网络预测的干净数据。**

从几何角度看，这是一个非常自然的速度场参数化。

但要注意：

$$
x_{\mathrm{next}}=D_\theta(x_t;t)
$$

只是一步 Euler 的结果，并不代表 ODE 的精确解必然等于当前去噪器输出。

### 4.2 为什么有利于数值积分？

ODE 求解器的误差与轨迹曲率密切相关。

如果轨迹在某个时间坐标下比较平直，那么使用较大的积分步长仍可能获得不错的精度。

论文通过一维双峰数据分布的实验，比较了 VP、VE 与 DDIM/EDM 参数化下的 ODE 轨迹。

结果发现，选择：

$$
\boxed{\sigma(t)=t,\qquad s(t)=1}
$$

能够让轨迹在较大和较小噪声区域更接近直线，主要曲率集中在中间噪声区域。

这有利于设计更高效的固定步长采样器。

这是论文基于轨迹分析和实验得到的设计判断，而不是证明它对任意数据分布都是最优时间参数化。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

## 五、论文公式 (5)：噪声时间表为什么是幂函数？

确定了 ODE：

$$
\frac{dx}{d\sigma}
=
\frac{x-D_\theta(x;\sigma)}{\sigma}
$$

接下来需要选择离散噪声水平：

$$
\sigma_0>\sigma_1>\cdots>\sigma_N=0
$$

EDM 希望低噪声区域使用较小的步长。

论文假设：

$$
\boxed{
\sigma_i=(Ai+B)^\rho
}
$$

其中：

- $i$：采样步编号。
- $A,B$：待确定的常数。
- $\rho>0$：控制步长分布的参数。

为什么可以写成这种形式？

这是作者选定的一个**可调参数化族**，不是从扩散概率模型唯一推导出来的。

现在要求：

$$
\sigma_0=\sigma_{\max}
$$

$$
\sigma_{N-1}=\sigma_{\min}
$$

将第一个条件代入：

$$
B^\rho=\sigma_{\max}
$$

因此：

$$
B=\sigma_{\max}^{1/\rho}
$$

第二个条件：

$$
[A(N-1)+B]^\rho=\sigma_{\min}
$$

两边取 $1/\rho$ 次幂：

$$
A(N-1)+B=\sigma_{\min}^{1/\rho}
$$

因此：

$$
A=
\frac{
\sigma_{\min}^{1/\rho}-\sigma_{\max}^{1/\rho}
}{
N-1
}
$$

代回：

$$
\boxed{
\sigma_i=
\left[
\sigma_{\max}^{1/\rho}
+
\frac{i}{N-1}
\left(
\sigma_{\min}^{1/\rho}
-
\sigma_{\max}^{1/\rho}
\right)
\right]^\rho
}
\tag{5}
$$

最后额外定义：

$$
\sigma_N=0
$$

### 5.1 为什么 $\rho$ 会影响步长？

把离散编号暂时看作连续变量：

$$
\sigma(i)=(Ai+B)^\rho
$$

对 $i$ 求导：

$$
\frac{d\sigma}{di}
=
\rho A(Ai+B)^{\rho-1}
$$

由于：

$$
A<0
$$

因此噪声水平随着 $i$ 增加而下降。

当：

$$
\rho>1
$$

时：

$$
\left|\frac{d\sigma}{di}\right|
=
\rho|A|\sigma^{(\rho-1)/\rho}
$$

所以随着 $\sigma$ 减小：

$$
\left|\frac{d\sigma}{di}\right|
$$

也逐渐减小。

也就是说，**低噪声区域的步长更小**。

### 5.2 为什么最终选择 $\rho=7$？

这里有一个容易忽略的论文细节。

作者在附录 D.1 的数值误差分析中发现：

$$
\rho\approx3
$$

能够使不同积分步的截断误差大致均衡。

但实际使用 FID 评估生成质量时：

$$
5\leq\rho\leq10
$$

通常表现更好。

最终选择：

$$
\boxed{\rho=7}
$$

这意味着：

**数值积分误差均衡，并不等于生成质量最优。**

尤其是在靠近数据分布的低噪声区域，积分误差可能对最终样本质量产生更大的影响。

因此作者愿意在低噪声区域投入更多积分步数。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

## 六、论文 Algorithm 1：Heun 二阶求解器完整推导

我们定义：

$$
f(x,\sigma)
=
\frac{x-D_\theta(x;\sigma)}{\sigma}
$$

需要求解：

$$
\frac{dx}{d\sigma}=f(x,\sigma)
$$

从：

$$
\sigma_i
$$

积分到：

$$
\sigma_{i+1}
$$

定义：

$$
h=\sigma_{i+1}-\sigma_i
$$

注意：

$$
h<0
$$

因为生成过程在降低噪声水平。

### 6.1 Euler 的推导

根据泰勒展开：

$$
x(\sigma+h)
=
x(\sigma)
+
h\frac{dx}{d\sigma}
+
\frac{h^2}{2}\frac{d^2x}{d\sigma^2}
+
O(h^3)
$$

Euler 只保留前两项：

$$
\boxed{
x_{i+1}
=
x_i+h f(x_i,\sigma_i)
}
$$

因此单步局部截断误差为：

$$
O(h^2)
$$

### 6.2 Heun 为什么是二阶？

先定义：

$$
k_1=f(x_i,\sigma_i)
$$

Euler 预测：

$$
\tilde x_{i+1}=x_i+hk_1
$$

然后在预测终点计算：

$$
k_2=f(\tilde x_{i+1},\sigma_{i+1})
$$

最终：

$$
\boxed{
x_{i+1}
=
x_i+\frac h2(k_1+k_2)
}
$$

现在证明它为什么比 Euler 更准确。

对 $k_2$ 进行多元泰勒展开：

$$
k_2
=
f(x_i+hk_1,\sigma_i+h)
$$

在 $(x_i,\sigma_i)$ 附近：

$$
k_2
=
f(x_i,\sigma_i)
+
h\frac{\partial f}{\partial\sigma}
+
h\frac{\partial f}{\partial x}k_1
+
O(h^2)
$$

其中：

$$
\frac{\partial f}{\partial x}
$$

表示速度场关于状态的 Jacobian。

由于：

$$
k_1=f(x_i,\sigma_i)
$$

所以：

$$
k_2
=
f+
h\left(
f_\sigma+f_xf
\right)
+
O(h^2)
$$

代入 Heun：

$$
x_{i+1}
=
x_i+\frac h2
\left[
2f+h(f_\sigma+f_xf)+O(h^2)
\right]
$$

整理：

$$
\boxed{
x_{i+1}
=
x_i+hf+
\frac{h^2}{2}(f_\sigma+f_xf)
+
O(h^3)
}
$$

另一方面，根据 ODE：

$$
x'=f(x,\sigma)
$$

对 $\sigma$ 求导：

$$
x''=f_\sigma+f_xx'
$$

因为：

$$
x'=f
$$

所以：

$$
\boxed{
x''=f_\sigma+f_xf
}
$$

因此精确解的泰勒展开：

$$
x(\sigma+h)
=
x(\sigma)+hf+
\frac{h^2}{2}(f_\sigma+f_xf)+O(h^3)
$$

与 Heun 的展开一致。

所以：

$$
\boxed{
\text{Heun 单步局部截断误差}=O(h^3)
}
$$

在通常的光滑性与稳定性条件下，全局误差为二阶。

### 6.3 为什么最后一步不使用 Heun？

EDM 的速度场：

$$
f(x,\sigma)
=
\frac{x-D_\theta(x;\sigma)}{\sigma}
$$

在：

$$
\sigma=0
$$

处不能直接代入。

因此当目标时间为：

$$
\sigma_{i+1}=0
$$

时，不能按常规方式计算终点速度：

$$
k_2=f(\tilde x_{i+1},0)
$$

所以论文最后一步使用 Euler。

如果总共进行 $N$ 次状态更新，前 $N-1$ 步各需要两次网络推理，最后一步需要一次：

$$
\boxed{
\mathrm{NFE}=2(N-1)+1=2N-1
}
$$

这是论文 Algorithm 1 的具体设计。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

## 七、论文第 4 节：随机采样的理论基础

这一节不能只理解成“EDM 又加了一点噪声”。

作者真正研究的是：

**既然 Probability Flow ODE 和反向 SDE 理论上具有相同的边缘分布，为什么实际使用随机采样时，生成质量有时反而更好？**

论文先构造一个广义随机动力系统，再提出 Stochastic Churn。

### 7.1 从 Langevin Dynamics 开始

考虑固定噪声水平 $\sigma$ 下的目标分布：

$$
p(x;\sigma)
$$

Langevin Dynamics：

$$
\boxed{
dx=
a\nabla_x\log p(x;\sigma)\,d\tau
+
\sqrt{2a}\,dW_\tau
}
$$

其中：

- $\tau$：Langevin 演化时间。
- $a>0$：控制演化速度的系数。
- $W_\tau$：标准布朗运动。

为什么它能够保持目标分布？

对应的 Fokker–Planck 方程为：

$$
\frac{\partial q}{\partial\tau}
=
-\nabla\cdot
\left(
a q\nabla\log p
\right)
+
a\Delta q
$$

如果当前：

$$
q=p
$$

则：

$$
-\nabla\cdot
\left(
a p\nabla\log p
\right)
=
-a\Delta p
$$

所以：

$$
\frac{\partial q}{\partial\tau}
=
-a\Delta p+a\Delta p=0
$$

因此：

$$
\boxed{
q=p
}
$$

是 Langevin Dynamics 的不变分布。

这意味着：**在理想 Score 下，Langevin 过程可以在不改变目标分布的前提下，持续更新样本。**

### 7.2 EDM 的广义 SDE：公式 (6)

EDM 把 Probability Flow ODE 与 Langevin 动力学组合。

论文写成：

$$
\boxed{
\begin{aligned}
dx_\pm
={}&
-\dot\sigma(t)\sigma(t)
\nabla_x\log p(x;\sigma(t))\,dt\\
&\pm
\beta(t)\sigma(t)^2
\nabla_x\log p(x;\sigma(t))\,dt\\
&+
\sqrt{2\beta(t)}\sigma(t)\,dW_t
\end{aligned}
}
\tag{6}
$$

其中：

$$
\beta(t)\geq0
$$

控制随机性的强度。

这里的 $+$ 和 $-$ 对应正向、反向时间演化的不同约定；使用反向时间时，随机微分和漂移符号必须一起处理，不能简单把 $dt<0$ 代入正向 SDE。

我们可以理解成：

$$
\boxed{
\text{广义 SDE}
=
\text{Probability Flow}
+
\text{Langevin 型随机演化}
}
$$

当：

$$
\beta(t)=0
$$

时：

$$
dx=
-\dot\sigma\sigma\nabla\log p\,dt
$$

退化为确定性 ODE。

当：

$$
\beta(t)>0
$$

时，过程会引入新的随机噪声。

### 7.3 为什么这种随机性可能有帮助？

理论上，如果 Score 完全准确、ODE/SDE 精确求解，两者都可以得到正确的目标边缘分布。

但实际情况是：

$$
D_\theta(x;\sigma)\neq D^*(x;\sigma)
$$

并且数值求解器使用有限步长。

因此，采样状态可能偏离理想边缘分布。

Langevin 型随机演化具有向目标密度重新调整的作用，所以适当随机性有可能缓解部分采样误差。

但是这种效果不是无条件成立的。

论文实验还发现，随机性过强可能造成细节丢失，甚至使样本颜色出现异常偏移。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

## 八、论文 Algorithm 2：Stochastic Churn 为什么这样构造？

作者没有直接采用通用的 SDE 数值求解器，而是设计了两阶段更新：

$$
\boxed{
\text{先提高噪声水平}
\rightarrow
\text{再使用 ODE 去噪}
}
$$

假设当前状态：

$$
x_i\sim p(x;\sigma_i)
$$

### 8.1 第一步：人为提高噪声水平

定义：

$$
\boxed{
\hat\sigma_i=(1+\gamma_i)\sigma_i
}
$$

其中：

$$
\gamma_i\geq0
$$

控制噪声增加量。

现在需要构造：

$$
\hat x_i\sim p(x;\hat\sigma_i)
$$

因为：

$$
x_i=y+\sigma_i\epsilon
$$

如果额外加入：

$$
\sqrt{\hat\sigma_i^2-\sigma_i^2}\,z
$$

其中：

$$
z\sim\mathcal N(0,I)
$$

并且独立于原有噪声，那么：

$$
\hat x_i
=
y+\sigma_i\epsilon+
\sqrt{\hat\sigma_i^2-\sigma_i^2}\,z
$$

由于两个噪声独立：

$$
\operatorname{Var}(\hat x_i\mid y)
=
\sigma_i^2I+
(\hat\sigma_i^2-\sigma_i^2)I
$$

得到：

$$
\boxed{
\operatorname{Var}(\hat x_i\mid y)
=
\hat\sigma_i^2I
}
$$

因此：

$$
\boxed{
\hat x_i
=
x_i+
\sqrt{\hat\sigma_i^2-\sigma_i^2}\,z
}
$$

能够把边缘分布从：

$$
p(x;\sigma_i)
$$

转变为：

$$
p(x;\hat\sigma_i)
$$

这里假设输入状态本身具有正确的边缘分布。

如果网络或数值求解已经产生偏差，单纯加噪并不能保证完全修正偏差。

### 8.2 第二步：从提高后的噪声水平重新去噪

现在：

$$
\hat x_i
$$

对应：

$$
\hat\sigma_i
$$

于是计算：

$$
d_i=
\frac{
\hat x_i-D_\theta(\hat x_i;\hat\sigma_i)
}{
\hat\sigma_i
}
$$

然后从：

$$
\hat\sigma_i
$$

积分到：

$$
\sigma_{i+1}
$$

先 Euler 预测：

$$
\tilde x_{i+1}
=
\hat x_i+
(\sigma_{i+1}-\hat\sigma_i)d_i
$$

再使用 Heun 修正：

$$
\boxed{
x_{i+1}
=
\hat x_i+
\frac{\sigma_{i+1}-\hat\sigma_i}{2}
(d_i+\tilde d_{i+1})
}
$$

其中：

$$
\tilde d_{i+1}
=
\frac{
\tilde x_{i+1}-D_\theta(\tilde x_{i+1};\sigma_{i+1})
}{
\sigma_{i+1}
}
$$

最后一步仍然使用 Euler。

### 8.3 为什么要限制 $\gamma_i$？

论文设置：

$$
\boxed{
\gamma_i=
\begin{cases}
\min\left(
\frac{S_{\mathrm{churn}}}{N},
\sqrt2-1
\right),
&
\sigma_i\in[S_{\min},S_{\max}]
\\
0,&\text{其他情况}
\end{cases}
}
$$

其中：

- $S_{\mathrm{churn}}$：控制总随机强度。
- $N$：采样步数。
- $S_{\min},S_{\max}$：启用随机性的噪声区间。

为什么上限是：

$$
\sqrt2-1
$$

因为：

$$
\hat\sigma_i=(1+\gamma_i)\sigma_i
$$

当：

$$
\gamma_i\leq\sqrt2-1
$$

有：

$$
\hat\sigma_i^2\leq2\sigma_i^2
$$

所以额外加入的噪声方差满足：

$$
\hat\sigma_i^2-\sigma_i^2
\leq\sigma_i^2
$$

也就是说，**单次额外注入的噪声方差不会超过当前噪声方差。**

这是一个有明确数学含义的经验约束。

论文还引入 $S_{\mathrm{noise}}$，允许对新注入的噪声做额外缩放。当它不等于 1 时，严格的方差匹配也随之改变，因此属于经验修正。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

# 九、论文第 5 节：Preconditioning 的完整数学推导

这一节是 EDM 最值得深入理解的训练贡献。

我们已经知道，理想去噪器为：

$$
D^*(x;\sigma)=\mathbb E[y\mid x]
$$

最直接的方法是让网络：

$$
F_\theta(x,\sigma)
$$

预测：

$$
y
$$

但论文认为，这不是最合理的参数化方式。

原因是：

$$
x=y+\sigma\epsilon
$$

随着 $\sigma$ 改变，输入的尺度可能变化很大。

例如：

- $\sigma$ 很小时，输入几乎等于真实数据。
- $\sigma$ 很大时，输入几乎完全由噪声主导。

如果网络直接预测 $y$，它需要在不同噪声水平下适应非常不同的输入统计特性。

而如果网络直接预测标准高斯噪声：

$$
\epsilon
$$

又存在另一个问题。

因为：

$$
\hat y=x-\sigma\epsilon_\theta
$$

当 $\sigma$ 很大时，噪声预测误差：

$$
\Delta\epsilon
$$

会导致干净数据预测误差：

$$
\boxed{
\Delta y=-\sigma\Delta\epsilon
}
$$

因此误差会被 $\sigma$ 放大。

EDM 希望找到一种统一参数化，使网络在不同噪声水平下都具有稳定的输入和目标尺度。

## 9.1 论文公式 (7)

作者定义：

$$
\boxed{
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}(\sigma)x
+
c_{\mathrm{out}}(\sigma)
F_\theta
\left(
c_{\mathrm{in}}(\sigma)x,
c_{\mathrm{noise}}(\sigma)
\right)
}
\tag{7}
$$

这里有四个系数：

| 系数 | 作用 |
|---|---|
| $c_{\mathrm{in}}$ | 让网络输入具有稳定的尺度 |
| $c_{\mathrm{skip}}$ | 决定直接保留多少输入 |
| $c_{\mathrm{out}}$ | 调整网络预测残差的尺度 |
| $c_{\mathrm{noise}}$ | 将噪声水平转换为网络条件 |

注意：

$$
D_\theta
$$

表示完整的去噪器。

而：

$$
F_\theta
$$

表示真正训练的神经网络。

**EDM 不是简单地让 $F_\theta$ 直接预测 $x_0$，而是让它预测一个经过预条件处理的目标。**

下面推导这些系数。

---

## 十、推导 $c_{\mathrm{in}}$：输入为什么要除以总标准差？

假设真实数据已经中心化：

$$
\mathbb E[y]=0
$$

并且各维度具有相同的典型方差：

$$
\operatorname{Var}(y)=\sigma_{\mathrm{data}}^2
$$

噪声：

$$
n\sim\mathcal N(0,\sigma^2I)
$$

输入：

$$
x=y+n
$$

由于数据和噪声独立：

$$
\operatorname{Var}(x)
=
\operatorname{Var}(y)+\operatorname{Var}(n)
$$

因此：

$$
\boxed{
\operatorname{Var}(x)
=
\sigma_{\mathrm{data}}^2+\sigma^2
}
$$

现在希望：

$$
c_{\mathrm{in}}x
$$

具有单位方差：

$$
\operatorname{Var}(c_{\mathrm{in}}x)=1
$$

所以：

$$
c_{\mathrm{in}}^2
(\sigma_{\mathrm{data}}^2+\sigma^2)=1
$$

得到：

$$
\boxed{
c_{\mathrm{in}}(\sigma)
=
\frac1{\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}}
}
$$

这一步很直接，本质就是标准化。

---

## 十一、推导 $c_{\mathrm{skip}}$：为什么它具有类似最优线性估计的形式？

现在考虑：

$$
D_\theta(x;\sigma)
=
c_{\mathrm{skip}}x+
c_{\mathrm{out}}F_\theta
$$

网络真正需要预测的部分是：

$$
y-c_{\mathrm{skip}}x
$$

如果希望网络的预测任务尽可能简单，就应该选择一个合适的 $c_{\mathrm{skip}}$，使剩余部分的方差尽可能小。

定义：

$$
r=y-c_{\mathrm{skip}}x
$$

由于：

$$
x=y+n
$$

所以：

$$
r
=
y-c_{\mathrm{skip}}(y+n)
$$

整理：

$$
\boxed{
r=
(1-c_{\mathrm{skip}})y-c_{\mathrm{skip}}n
}
$$

由于 $y,n$ 独立：

$$
\operatorname{Var}(r)
=
(1-c_{\mathrm{skip}})^2\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}^2\sigma^2
$$

现在希望最小化：

$$
J(c_{\mathrm{skip}})
=
(1-c_{\mathrm{skip}})^2\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}^2\sigma^2
$$

对 $c_{\mathrm{skip}}$ 求导：

$$
\frac{dJ}{dc_{\mathrm{skip}}}
=
-2(1-c_{\mathrm{skip}})\sigma_{\mathrm{data}}^2
+
2c_{\mathrm{skip}}\sigma^2
$$

令导数为零：

$$
-(1-c_{\mathrm{skip}})\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}\sigma^2=0
$$

展开：

$$
-\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}\sigma^2=0
$$

得到：

$$
c_{\mathrm{skip}}
(\sigma_{\mathrm{data}}^2+\sigma^2)
=
\sigma_{\mathrm{data}}^2
$$

所以：

$$
\boxed{
c_{\mathrm{skip}}(\sigma)
=
\frac{\sigma_{\mathrm{data}}^2}
{\sigma_{\mathrm{data}}^2+\sigma^2}
}
$$

这个公式有两个重要极限。

当：

$$
\sigma\rightarrow0
$$

有：

$$
c_{\mathrm{skip}}\rightarrow1
$$

说明几乎不需要去噪。

当：

$$
\sigma\rightarrow\infty
$$

有：

$$
c_{\mathrm{skip}}\rightarrow0
$$

说明输入主要是噪声，不应该直接保留。

这里还有一个重要细节：

**这个系数是最优线性残差缩放系数，不代表真实数据分布必须是高斯分布，也不代表完整的最优去噪器一定是线性的。**

非线性的剩余预测任务仍然由 $F_\theta$ 完成。

---

## 十二、推导 $c_{\mathrm{out}}$：为什么要标准化残差？

前面得到：

$$
r=y-c_{\mathrm{skip}}x
$$

现在希望网络的目标：

$$
\frac r{c_{\mathrm{out}}}
$$

具有单位方差。

因此要求：

$$
c_{\mathrm{out}}^2
=
\operatorname{Var}(r)
$$

我们已经知道：

$$
\operatorname{Var}(r)
=
(1-c_{\mathrm{skip}})^2\sigma_{\mathrm{data}}^2
+
c_{\mathrm{skip}}^2\sigma^2
$$

令：

$$
A=\sigma_{\mathrm{data}}^2,\qquad B=\sigma^2
$$

那么：

$$
c_{\mathrm{skip}}=\frac A{A+B}
$$

所以：

$$
1-c_{\mathrm{skip}}=\frac B{A+B}
$$

代入：

$$
\operatorname{Var}(r)
=
\frac{B^2}{(A+B)^2}A
+
\frac{A^2}{(A+B)^2}B
$$

合并：

$$
=
\frac{AB^2+A^2B}{(A+B)^2}
$$

提取 $AB$：

$$
=
\frac{AB(A+B)}{(A+B)^2}
$$

得到：

$$
\boxed{
\operatorname{Var}(r)=\frac{AB}{A+B}
}
$$

恢复原来的符号：

$$
\operatorname{Var}(r)
=
\frac{
\sigma_{\mathrm{data}}^2\sigma^2
}{
\sigma_{\mathrm{data}}^2+\sigma^2
}
$$

所以：

$$
\boxed{
c_{\mathrm{out}}(\sigma)
=
\frac{
\sigma_{\mathrm{data}}\sigma
}{
\sqrt{\sigma_{\mathrm{data}}^2+\sigma^2}
}
}
$$

这样，网络的实际监督目标：

$$
\boxed{
F_{\mathrm{target}}
=
\frac{
y-c_{\mathrm{skip}}x
}{
c_{\mathrm{out}}
}
}
$$

在上述统计假设下具有单位方差。

---

## 十三、公式 (8)：为什么 EDM 的损失权重是这个形式？

论文训练目标：

$$
\mathcal L
=
\mathbb E_{\sigma,y,n}
\left[
\lambda(\sigma)
\|D_\theta(y+n;\sigma)-y\|^2
\right]
$$

令：

$$
x=y+n
$$

代入：

$$
D_\theta
=
c_{\mathrm{skip}}x+
c_{\mathrm{out}}F_\theta
$$

得到：

$$
\mathcal L
=
\mathbb E
\left[
\lambda(\sigma)
\|
c_{\mathrm{skip}}x+
c_{\mathrm{out}}F_\theta-y
\|^2
\right]
$$

把括号重新整理：

$$
c_{\mathrm{skip}}x+
c_{\mathrm{out}}F_\theta-y
=
c_{\mathrm{out}}
\left[
F_\theta-
\frac{y-c_{\mathrm{skip}}x}{c_{\mathrm{out}}}
\right]
$$

所以：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\lambda(\sigma)c_{\mathrm{out}}^2
\left\|
F_\theta-F_{\mathrm{target}}
\right\|^2
\right]
}
\tag{8}
$$

这就是论文公式 (8) 的核心内容。

注意这里出现了：

$$
\boxed{
\lambda(\sigma)c_{\mathrm{out}}^2
}
$$

它才是作用于原始网络 $F_\theta$ 的有效损失权重。

EDM 希望：

$$
\lambda(\sigma)c_{\mathrm{out}}^2=1
$$

因此：

$$
\boxed{
\lambda(\sigma)=\frac1{c_{\mathrm{out}}^2}
}
$$

代入：

$$
c_{\mathrm{out}}^2
=
\frac{
\sigma_{\mathrm{data}}^2\sigma^2
}{
\sigma_{\mathrm{data}}^2+\sigma^2
}
$$

得到：

$$
\boxed{
\lambda(\sigma)
=
\frac{
\sigma_{\mathrm{data}}^2+\sigma^2
}{
\sigma_{\mathrm{data}}^2\sigma^2
}
}
$$

至此，EDM 的主要预条件公式就全部推导出来了。

最后：

$$
\boxed{
c_{\mathrm{noise}}(\sigma)=\frac14\ln\sigma
}
$$

是作者采用的经验噪声条件缩放，不是通过前面的方差最优化推导出来的。

这一组推导对应论文第 5 节、公式 (7)、(8) 和附录 B.6。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

# 十四、为什么 EDM 要采用对数正态噪声采样？

EDM 的训练噪声分布为：

$$
\boxed{
\ln\sigma\sim
\mathcal N(P_{\mathrm{mean}},P_{\mathrm{std}}^2)
}
$$

等价于：

$$
\boxed{
\sigma=
\exp(P_{\mathrm{mean}}+P_{\mathrm{std}}z)
}
$$

其中：

$$
z\sim\mathcal N(0,1)
$$

典型配置：

$$
P_{\mathrm{mean}}=-1.2
$$

$$
P_{\mathrm{std}}=1.2
$$

为什么不均匀采样 $\sigma$？

因为作者发现，不同噪声水平的学习收益并不相同。

在极低噪声水平：

$$
x\approx y
$$

网络的去噪任务相对简单，而且剩余噪声很小。

在极高噪声水平：

$$
x\approx\sigma\epsilon
$$

输入与真实数据的关联很弱，最优去噪器逐渐接近数据均值。

中间噪声水平通常包含更值得学习的结构信息。

因此作者使用对数正态分布，将更多训练资源分配给中间噪声区域。

这里必须区分：

$$
\lambda(\sigma)
$$

和：

$$
p_{\mathrm{train}}(\sigma)
$$

前者控制单个噪声水平的损失权重。

后者控制该噪声水平被采样的频率。

它们是两个独立设计变量。

EDM 同时优化了这两者，而不是只修改损失权重。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

# 十五、论文实验部分：哪些改进真正有效？

论文第 3–5 节的实验主要围绕两个问题：

**第一个问题：不重新训练网络，只修改采样器，能提升多少？**

作者使用已有的 VP、VE 和 iDDPM 模型，比较原始采样器与 EDM 的采样方案。

主要发现是：

- 二阶 Heun 通常比 Euler 具有更好的 NFE—生成质量权衡。
- 合理的噪声时间表能够显著减少所需网络推理次数。
- 选择 $\sigma(t)=t,s(t)=1$ 有助于改善 ODE 轨迹的数值特性。
- 这些采样改进可以用于已有预训练模型，不必重新训练网络。

论文报告，在对应实验设置下，达到相近生成质量所需的 NFE 分别降低约 7.3 倍、300 倍和 3.2 倍。不同模型的提升幅度差异很大。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

**第二个问题：重新训练网络时，哪些设计最重要？**

论文 Table 2 按照 A–F 六组配置逐步进行消融：

| 配置 | 改进 |
|---|---|
| A | 原始 VP/VE 训练配置 |
| B | 重新调整基础超参数 |
| C | 重新分配网络容量 |
| D | 使用 EDM 网络预条件 |
| E | 使用 EDM 损失权重和噪声采样分布 |
| F | 增加不泄漏的数据增强 |

其中有一个值得注意的现象：

**单独更换网络预条件，不一定立刻显著改善 FID。**

它的重要作用之一，是使不同噪声水平的训练更加稳定，为后续优化损失权重和噪声分布创造条件。

当作者进一步引入 EDM 的损失权重和噪声采样策略时，多个实验设置的 FID 才出现明显改善。

这说明：

$$
\boxed{
\text{Preconditioning}
+
\text{Loss Weighting}
+
\text{Noise Sampling}
}
$$

需要作为相互配合的训练设计来理解，而不是把每一项孤立地看作必然提升生成质量的技巧。[NeurIPS Papers](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf?utm_source=chatgpt.com)

---

## 十六、为什么论文还研究数据增强？

论文第 5 节还讨论了数据增强正则化。

传统图像生成模型可以通过旋转、翻转等方式增加训练数据的多样性。

但扩散模型存在一个问题：

如果直接对图像进行增强，而生成模型无法区分增强与原始数据，就可能在生成结果中保留不希望出现的增强特征。

因此，作者采用了带条件的数据增强：

$$
y'=A_a(y)
$$

其中：

- $A_a$：由增强参数 $a$ 决定的图像变换。
- $y'$：增强后的图像。

训练时不仅输入加噪图像，还向网络提供增强参数：

$$
D_\theta(x;\sigma,a)
$$

推理时使用表示不进行增强的条件：

$$
a=0
$$

从而让网络生成未增强的数据分布。

这是借鉴 GAN 中非泄漏数据增强思想的一项训练改进。

对于自动驾驶轨迹规划，这个思路也有启发意义：如果对轨迹和场景进行几何变换，必须保证地图、障碍物、导航与轨迹之间的几何关系同步变化。

---

# 十七、论文附录的作用：哪些内容不能忽略？

EDM 的附录并不只是补充实验。

其中有不少重要理论细节。

**附录 B：数学推导**

包括：

- B.1：从已有的 SDE/ODE 理论出发。
- B.2：推导 EDM 的统一 ODE。
- B.3：证明去噪器与 Score 的关系。
- B.4：解释 ODE 数值求解。
- B.5：推导广义随机采样的 SDE。
- B.6：推导网络预条件系数。

我们前面已经详细展开了这些部分最核心的数学关系。

**附录 C：将已有模型重新表示到 EDM 框架**

这部分非常重要，因为它说明 EDM 不只是提出一种新模型。

它还给出了 VP、VE、iDDPM/DDIM 在统一框架下对应的：

$$
\sigma(t),\quad s(t),\quad c_{\mathrm{skip}},\quad
c_{\mathrm{out}},\quad c_{\mathrm{in}},\quad
c_{\mathrm{noise}}
$$

以及训练噪声分布和损失权重。

因此，理解 EDM Table 1 的意义，是理解论文统一框架的关键。

**附录 D：数值求解器分析**

包括采样时间表、积分误差以及二阶求解器的设计。

这里最重要的结论是：数值积分误差和最终生成质量不是完全等价的优化目标。

**附录 E、F：随机采样与训练实现细节**

包括随机性参数的实验分析、数据增强、网络架构与训练配置。

这些内容主要用于理解论文结果的可复现性和适用范围。

---

# 十八、EDM 和你之前研究的 Flow Matching 到底是什么关系？

最后把 EDM 放回你正在研究的轨迹规划问题中。

EDM 的概率路径：

$$
\boxed{
x_\sigma=y+\sigma\epsilon
}
$$

而常见 CondOT Flow Matching 使用：

$$
\boxed{
x_t=(1-t)\epsilon+ty
}
$$

其中：

$$
t\in[0,1]
$$

我们尝试建立联系。

对于：

$$
t>0
$$

把 CondOT 状态除以 $t$：

$$
\frac{x_t}{t}
=
y+\frac{1-t}{t}\epsilon
$$

定义：

$$
\boxed{
\sigma=\frac{1-t}{t}
}
$$

得到：

$$
\boxed{
\frac{x_t}{t}=y+\sigma\epsilon
}
$$

这与 EDM 的加性噪声形式相同。

因此，**在独立噪声—数据耦合、相应的状态缩放和时间重参数化下，常见 CondOT 路径可以与 EDM 的加性高斯噪声路径建立对应。**

但需要注意，两个模型的原始状态变量不同：

$$
x_t^{\mathrm{CondOT}}
=
t\,x_\sigma^{\mathrm{EDM}}
$$

而且它们通常采用不同的网络输出参数化和训练损失。

所以不能因为概率路径能够转换，就认为两种训练目标和有限步采样器自动完全等价。

这个联系对于你后续研究 DPM-Solver、Flow Matching 速度预测和 $x_0$ 预测非常重要。

---

# 十九、整篇 EDM 论文的最终总结

按照论文原本的逻辑，可以把主要贡献归纳为以下几个层次。

| 层次 | 核心设计 | 解决的问题 |
|---|---|---|
| 概率分布 | $p(x;\sigma)$ | 统一高斯加噪边缘分布 |
| ODE | $\dot x=-\dot\sigma\sigma\nabla\log p$ | 构造确定性生成过程 |
| 去噪器 | $D_\theta$ | 用数据预测代替直接计算 Score |
| 状态参数化 | $\sigma(t),s(t)$ | 统一 VP、VE、DDIM |
| 数值求解 | Heun | 降低积分误差 |
| 时间表 | $\rho$-schedule | 优化不同噪声区域的步长 |
| 随机采样 | Stochastic Churn | 引入可控随机性 |
| 网络预条件 | $c_{\mathrm{in}},c_{\mathrm{skip}},c_{\mathrm{out}}$ | 稳定输入与预测目标的尺度 |
| 训练损失 | $\lambda(\sigma)$ | 平衡不同噪声水平的有效损失 |
| 噪声采样 | 对数正态分布 | 调整训练资源在噪声水平上的分配 |
| 数据增强 | 增强参数条件化 | 改善泛化并减少增强泄漏 |

### 我认为这篇论文最值得深入理解的三个结论

**第一，扩散模型的连续概率路径、网络预测形式与数值求解器是可以分开研究的。**

这解释了为什么一个 DDPM 训练的网络可以配合 DDIM、Heun 或其他兼容的采样器。

**第二，EDM 的网络预条件不是简单的归一化，而是在不同噪声水平之间平衡预测任务。**

特别是：

$$
\boxed{
D_\theta
=
c_{\mathrm{skip}}x+
c_{\mathrm{out}}F_\theta
}
$$

它使网络能够根据噪声水平，在更接近预测数据和更接近预测噪声的目标之间平滑变化。

**第三，采样器的数值精度和生成质量需要分别评价。**

Heun 可以降低数值积分误差，但网络本身的 Score 估计误差、训练噪声分布、随机采样机制等仍会影响最终生成结果。

这也是为什么 EDM 不只研究求解器，还要系统研究训练和随机性。

---

**原论文：** [Karras et al., NeurIPS 2022](https://papers.nips.cc/paper/2022/file/a98846e9d9cc01cfb87eb694d946ce6b-Paper-Conference.pdf)

**官方实现：** [NVlabs/edm](https://github.com/NVlabs/edm)

